# Project: Question-Answering on Private Documents (RAG)

In [1]:
pip install -q -r ./requirements.txt # This notebook uses the versions of the libraries (OpenAI, LangChain, etc.) shown in requirements.txt.

Note: you may need to restart the kernel to use updated packages.


In [2]:
pip show langchain

Name: langchain
Version: 1.2.12
Summary: Building applications with LLMs through composability
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: /home/dda/Development/AI_LLM_Notebooks/venv313/lib/python3.13/site-packages
Requires: langchain-core, langgraph, pydantic
Required-by: 
Note: you may need to restart the kernel to use updated packages.


In [43]:
import os
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(), override=True)

True

## Loading Documents

In [44]:
# loading PDF, DOCX and TXT files as LangChain Documents
def load_document(file):
    import os
    name, extension = os.path.splitext(file)

    if extension == '.pdf':
        # Moved from langchain.document_loaders to langchain_community.document_loaders (v0.1.0+)
        from langchain_community.document_loaders import PyPDFLoader
        print(f'Loading {file}')
        loader = PyPDFLoader(file)
    elif extension == '.docx':
        # Moved from langchain.document_loaders to langchain_community.document_loaders (v0.1.0+)
        from langchain_community.document_loaders import Docx2txtLoader
        print(f'Loading {file}')
        loader = Docx2txtLoader(file)
    elif extension == '.txt':
        # Moved from langchain.document_loaders to langchain_community.document_loaders (v0.1.0+)
        from langchain_community.document_loaders import TextLoader
        loader = TextLoader(file)
    else:
        print('Document format is not supported!')
        return None

    data = loader.load()
    return data


# wikipedia
def load_from_wikipedia(query, lang='en', load_max_docs=2):
    # Moved from langchain.document_loaders to langchain_community.document_loaders (v0.1.0+)
    from langchain_community.document_loaders import WikipediaLoader
    loader = WikipediaLoader(query=query, lang=lang, load_max_docs=load_max_docs)
    data = loader.load()
    return data


## Chunking Data

In [45]:
def chunk_data(data, chunk_size=256):
    # Moved from langchain.text_splitter to langchain_text_splitters (v0.2.0+)
    from langchain_text_splitters import RecursiveCharacterTextSplitter
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=0)
    chunks = text_splitter.split_documents(data)
    return chunks


## Calculating Cost

In [46]:
def print_embedding_cost(texts):
    import tiktoken
    enc = tiktoken.encoding_for_model('text-embedding-3-small')
    total_tokens = sum([len(enc.encode(page.page_content)) for page in texts])
    print(f'Total Tokens: {total_tokens}')
    print(f'Embedding Cost in USD: {total_tokens / 1000 * 0.00002:.6f}')

## Embedding and Uploading to a Vector Database (Pinecone)

In [47]:
def insert_or_fetch_embeddings(index_name, chunks):
    import pinecone
    # langchain_community.vectorstores.Pinecone deprecated and incompatible with pinecone v7+
    # Use langchain_pinecone.PineconeVectorStore instead
    from langchain_pinecone import PineconeVectorStore
    from langchain_openai import OpenAIEmbeddings
    from pinecone import PodSpec

    pc = pinecone.Pinecone()

    embeddings = OpenAIEmbeddings(model='text-embedding-3-small', dimensions=1536)  # 512 works as well

    # loading from existing index
    if index_name in pc.list_indexes():
        print(f'Index {index_name} already exists. Loading embeddings ... ', end='')
        vector_store = PineconeVectorStore.from_existing_index(index_name, embeddings)
        print('Ok')
    else:
        # creating the index and embedding the chunks into the index
        print(f'Creating index {index_name} and embeddings ...', end='')

        # creating a new index
        pc.create_index(
            name=index_name,
            dimension=1536,
            metric='cosine',
            spec=PodSpec(
                environment='gcp-starter'
            )
        )

        # processing the input documents, generating embeddings using the provided `OpenAIEmbeddings` instance,
        # inserting the embeddings into the index and returning a new PineconeVectorStore object.
        vector_store = PineconeVectorStore.from_documents(chunks, embeddings, index_name=index_name)
        print('Ok')

    return vector_store


In [48]:
def delete_pinecone_index(index_name='all'):
    import pinecone
    pc = pinecone.Pinecone()
    
    if index_name == 'all':
        indexes = pc.list_indexes().names()
        print('Deleting all indexes ... ')
        for index in indexes:
            pc.delete_index(index)
        print('Ok')
    else:
        print(f'Deleting index {index_name} ...', end='')
        pc.delete_index(index_name)
        print('Ok')
    

## Asking and Getting Answers

In [49]:
def ask_and_get_answer(vector_store, q, k=3):
    # RetrievalQA removed in langchain v1.0+; replaced with LCEL chain
    from langchain_core.prompts import ChatPromptTemplate
    from langchain_core.output_parsers import StrOutputParser
    from langchain_core.runnables import RunnablePassthrough
    from langchain_openai import ChatOpenAI

    llm = ChatOpenAI(model='gpt-5.2', temperature=1)
    retriever = vector_store.as_retriever(search_type='similarity', search_kwargs={'k': k})

    def format_docs(docs):
        return "\n\n".join(doc.page_content for doc in docs)

    prompt = ChatPromptTemplate.from_template(
        "Answer the question based only on the following context:\n\n{context}\n\nQuestion: {question}"
    )

    # LCEL chain: retrieve context → format → prompt → LLM → parse output
    chain = (
        {"context": retriever | format_docs, "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )

    answer = chain.invoke(q)
    return {"query": q, "result": answer}

## Using Chroma as a Vector DB

In [50]:
# pip install -q chromadb

In [51]:
def create_embeddings_chroma(chunks, persist_directory='./chroma_db'):
    # Moved from langchain.vectorstores to langchain_community.vectorstores (v0.1.0+)
    # from langchain_community.vectorstores import Chroma
    from langchain_chroma import Chroma
    from langchain_openai import OpenAIEmbeddings

    # Instantiate an embedding model from OpenAI (smaller version for efficiency)
    embeddings = OpenAIEmbeddings(model='text-embedding-3-small', dimensions=1536)

    # Create a Chroma vector store using the provided text chunks and embedding model,
    # configuring it to save data to the specified directory
    vector_store = Chroma.from_documents(chunks, embeddings, persist_directory=persist_directory)

    return vector_store  # Return the created vector store


In [52]:
def load_embeddings_chroma(persist_directory='./chroma_db'):
    # Moved from langchain.vectorstores to langchain_community.vectorstores (v0.1.0+)
    # from langchain_community.vectorstores import Chroma
    from langchain_chroma import Chroma
    from langchain_openai import OpenAIEmbeddings

    # Instantiate the same embedding model used during creation
    embeddings = OpenAIEmbeddings(model='text-embedding-3-small', dimensions=1536)

    # Load a Chroma vector store from the specified directory, using the provided embedding function
    vector_store = Chroma(persist_directory=persist_directory, embedding_function=embeddings)

    return vector_store  # Return the loaded vector store

#### Running Code

In [53]:
# Loading the pdf document into LangChain 
data = load_document('files/rag_powered_by_google_search.pdf')

# Splitting the document into chunks
chunks = chunk_data(data, chunk_size=256)

# Creating a Chroma vector store using the provided text chunks and embedding model (default is text-embedding-3-small)
vector_store = create_embeddings_chroma(chunks)

Loading files/rag_powered_by_google_search.pdf


In [54]:
# Asking questions
q = 'What is Vertex AI Search?'
answer = ask_and_get_answer(vector_store, q)
print(answer)

{'query': 'What is Vertex AI Search?', 'result': 'Vertex AI Search is a Vertex AI search offering that provides customizable answers, search tuning, vector search, and grounding and compliance updates designed for enterprises.'}


In [55]:
print(answer['result'])

Vertex AI Search is a Vertex AI search offering that provides customizable answers, search tuning, vector search, and grounding and compliance updates designed for enterprises.


In [56]:
# Load a Chroma vector store from the specified directory (default ./chroma_db) 
db = load_embeddings_chroma()
q = 'How many pairs of questions and answers had the StackOverflow dataset?'
answer = ask_and_get_answer(db, q)
print(answer)

{'query': 'How many pairs of questions and answers had the StackOverflow dataset?', 'result': 'The StackOverflow dataset had **8 million** pairs of questions and answers.'}


In [57]:
# We can't ask follow-up questions. There is no memory (chat history) available.
q = 'Multiply that number by 2.'
answer = ask_and_get_answer(vector_store, q)
print(answer['result'])

3 × 2 = 6


## Adding Memory (Chat History)

In [58]:
from langchain_openai import ChatOpenAI
# ConversationalRetrievalChain and ConversationBufferMemory removed in langchain v1.0+
# Replaced with LCEL + RunnableWithMessageHistory
# langchain.prompts moved to langchain_core.prompts
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory

llm = ChatOpenAI(model_name='gpt-5.2', temperature=0)
retriever = vector_store.as_retriever(search_type='similarity', search_kwargs={'k': 5})

# Initialize/reset session store for a fresh conversation
store = {}


def get_session_history(session_id: str) -> InMemoryChatMessageHistory:
    """Return (or create) the in-memory chat history for a given session."""
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]


prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer the user's questions based on the following context:\n\n{context}"),
    MessagesPlaceholder(variable_name="chat_history"),  # injected by RunnableWithMessageHistory
    ("human", "{question}"),
])


def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


# LCEL chain: attach retrieved context, then prompt → LLM → parse output
chain = (
    RunnablePassthrough.assign(
        context=lambda x: format_docs(retriever.invoke(x["question"]))
    )
    | prompt
    | llm
    | StrOutputParser()
)

# Wrap chain with message history for multi-turn conversation
crc = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="question",
    history_messages_key="chat_history",
)

In [59]:
# Session store for RunnableWithMessageHistory (replaces ConversationBufferMemory)
store = {}

def ask_question(q, chain, session_id="default"):
    # RunnableWithMessageHistory requires a session config for memory routing
    answer = chain.invoke(
        {"question": q},
        config={"configurable": {"session_id": session_id}}
    )
    history = store[session_id].messages if session_id in store else []
    return {"question": q, "answer": answer, "chat_history": history}


In [60]:
data = load_document('files/rag_powered_by_google_search.pdf')
chunks = chunk_data(data, chunk_size=256)
vector_store = create_embeddings_chroma(chunks)

Loading files/rag_powered_by_google_search.pdf


In [61]:
q = 'How many pairs of questions and answers had the StackOverflow dataset?'
result = ask_question(q, crc)
print(result)

{'question': 'How many pairs of questions and answers had the StackOverflow dataset?', 'answer': 'The StackOverflow dataset had **8 million pairs of questions and answers**.', 'chat_history': [HumanMessage(content='How many pairs of questions and answers had the StackOverflow dataset?', additional_kwargs={}, response_metadata={}), AIMessage(content='The StackOverflow dataset had **8 million pairs of questions and answers**.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]}


In [62]:
print(result['answer'])

The StackOverflow dataset had **8 million pairs of questions and answers**.


In [63]:
q = 'Multiply that number by 10.'
result = ask_question(q, crc)

In [64]:
print(result['answer'])

**80 million**.


In [65]:
q = 'Devide the result by 80.'
result = ask_question(q, crc)
print(result['answer'])


**1 million**.


In [66]:
for item in result['chat_history']:
    print(item)

content='How many pairs of questions and answers had the StackOverflow dataset?' additional_kwargs={} response_metadata={}
content='The StackOverflow dataset had **8 million pairs of questions and answers**.' additional_kwargs={} response_metadata={} tool_calls=[] invalid_tool_calls=[]
content='Multiply that number by 10.' additional_kwargs={} response_metadata={}
content='**80 million**.' additional_kwargs={} response_metadata={} tool_calls=[] invalid_tool_calls=[]
content='Devide the result by 80.' additional_kwargs={} response_metadata={}
content='**1 million**.' additional_kwargs={} response_metadata={} tool_calls=[] invalid_tool_calls=[]


### Loop for asking questions

In [25]:
while True:
    q = input('Your question: ')
    if q.lower() in 'exit quit bye':
        print('Bye bye!')
        break
    result = ask_question(q, crc)
    print(result['answer'])
    print('-' * 100)
    

Your question:  What is RAG?


RAG stands for Retrieve, Answer, Generate. It is a system that combines information retrieval and natural language generation to answer questions by retrieving relevant information from a large text corpus and generating responses.
----------------------------------------------------------------------------------------------------


Your question:  quit


Bye bye!


## Using a Custom Prompt

In [67]:
from langchain_openai import ChatOpenAI
# ConversationalRetrievalChain and ConversationBufferMemory removed in langchain v1.0+
# Replaced with LCEL + RunnableWithMessageHistory
# langchain.prompts moved to langchain_core.prompts
from langchain_core.prompts import (
    ChatPromptTemplate,
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate,
    MessagesPlaceholder,
)
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory

llm = ChatOpenAI(model_name='gpt-5.2', temperature=0)
retriever = vector_store.as_retriever(search_type='similarity', search_kwargs={'k': 5})

# Initialize/reset session store for a fresh conversation
store = {}

system_template = r'''
Use the following pieces of context to answer the user's question.
Before answering translate your response to Spanish.
If you don't find the answer in the provided context, just respond "I don't know."
---------------
Context: ```{context}```
'''

user_template = '''
Question: ```{question}```
'''

messages = [
    SystemMessagePromptTemplate.from_template(system_template),
    MessagesPlaceholder(variable_name="chat_history"),  # injected by RunnableWithMessageHistory
    HumanMessagePromptTemplate.from_template(user_template),
]

qa_prompt = ChatPromptTemplate.from_messages(messages)


def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


# LCEL chain with custom prompt: attach context, then prompt → LLM → parse output
chain = (
    RunnablePassthrough.assign(
        context=lambda x: format_docs(retriever.invoke(x["question"]))
    )
    | qa_prompt
    | llm
    | StrOutputParser()
)

# Wrap chain with message history for multi-turn conversation
crc = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="question",
    history_messages_key="chat_history",
)

In [68]:
print(qa_prompt)

input_variables=['chat_history', 'context', 'question'] input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag='ChatMessageChunk')], typing.Annotated[langchain_core.messages.system.SystemMessageChunk, Tag(tag='SystemMessageChunk')], typing.Annotated[langcha

In [69]:
# db = load_embeddings_chroma()
q = 'How many pairs of questions and answers had the StackOverflow dataset?'
result = ask_question(q, crc)
print(result)

{'question': 'How many pairs of questions and answers had the StackOverflow dataset?', 'answer': 'El conjunto de datos de StackOverflow tenía **8 millones** de pares de preguntas y respuestas.', 'chat_history': [HumanMessage(content='How many pairs of questions and answers had the StackOverflow dataset?', additional_kwargs={}, response_metadata={}), AIMessage(content='El conjunto de datos de StackOverflow tenía **8 millones** de pares de preguntas y respuestas.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]}


In [70]:
q = 'When was Elon Musk born?'
result = ask_question(q, crc)
print(result)

{'question': 'When was Elon Musk born?', 'answer': 'No lo sé.', 'chat_history': [HumanMessage(content='How many pairs of questions and answers had the StackOverflow dataset?', additional_kwargs={}, response_metadata={}), AIMessage(content='El conjunto de datos de StackOverflow tenía **8 millones** de pares de preguntas y respuestas.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='When was Elon Musk born?', additional_kwargs={}, response_metadata={}), AIMessage(content='No lo sé.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]}


In [71]:
q = 'When was Bill Gates born?'
result = ask_question(q, crc)
print(result)

{'question': 'When was Bill Gates born?', 'answer': 'No lo sé.', 'chat_history': [HumanMessage(content='How many pairs of questions and answers had the StackOverflow dataset?', additional_kwargs={}, response_metadata={}), AIMessage(content='El conjunto de datos de StackOverflow tenía **8 millones** de pares de preguntas y respuestas.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='When was Elon Musk born?', additional_kwargs={}, response_metadata={}), AIMessage(content='No lo sé.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='When was Bill Gates born?', additional_kwargs={}, response_metadata={}), AIMessage(content='No lo sé.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]}


In [72]:
print(result['answer'])

No lo sé.
